# Website Security Project – lokale Version mit Ollama/Mistral

Die Auswertung wird lokal über **Ollama** mit dem Modell **Mistral** erstellt.

> Wichtig: Verwenden Sie dieses Notebook nur für Webseiten, für die Sie eine ausdrückliche Berechtigung zur Analyse haben. Das Notebook führt keine aktiven Angriffe aus, sammelt aber HTTP-/Browser-Informationen und erstellt daraus einen Security-Report.


## 1. Voraussetzungen

Vor dem Start lokal ausführen:

```bash
ollama serve
ollama pull mistral
```

Zusätzlich muss ein lokaler Chrome- oder Chromium-Browser installiert sein. Selenium verwendet in aktuellen Versionen den integrierten Selenium Manager, um den passenden Treiber zu finden.


In [65]:
# Optional: Abhängigkeiten installieren
# Führen Sie diese Zelle einmalig aus, falls die Pakete noch nicht installiert sind.

%pip install -q selenium beautifulsoup4 requests tldextract ipython

Note: you may need to restart the kernel to use updated packages.


In [66]:
import json
import re
import time
from collections import Counter
from urllib.parse import urlparse, urljoin

import requests
import tldextract
from bs4 import BeautifulSoup
from IPython.display import display, Markdown

from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.common.exceptions import WebDriverException

OLLAMA_HOST = "http://127.0.0.1:11434"
OLLAMA_MODEL = "mistral"
REQUEST_TIMEOUT = 15
PAGE_LOAD_WAIT_SECONDS = 3
MAX_PROMPT_CHARS = 18000

print("Konfiguration geladen.")

Konfiguration geladen.


In [67]:
def check_ollama(host: str = OLLAMA_HOST, model: str = OLLAMA_MODEL) -> None:
    """Prüft, ob Ollama erreichbar ist und ob das gewünschte Modell vorhanden ist."""
    try:
        response = requests.get(f"{host}/api/tags", timeout=5)
        response.raise_for_status()
    except Exception as exc:
        raise RuntimeError(
            f"Ollama ist unter {host} nicht erreichbar. Starten Sie Ollama lokal mit: ollama serve"
        ) from exc

    models = response.json().get("models", [])
    model_names = {m.get("name", "") for m in models}
    available = any(name == model or name.startswith(model + ":") for name in model_names)
    if not available:
        raise RuntimeError(
            f"Das Modell '{model}' wurde in Ollama nicht gefunden. Installieren Sie es mit: ollama pull {model}"
        )

    print(f"Ollama ist erreichbar. Modell '{model}' ist verfügbar.")

check_ollama()

Ollama ist erreichbar. Modell 'mistral' ist verfügbar.


In [68]:
url = input("Zu analysierende URL eingeben, z.B. https://example.com: ").strip()

if not re.match(r"^https?://", url, flags=re.IGNORECASE):
    url = "https://" + url

parsed = urlparse(url)
base_url = f"{parsed.scheme}://{parsed.netloc}"
registered_domain = tldextract.extract(url).registered_domain

print(f"Ziel-URL: {url}")
print(f"Basis-URL: {base_url}")
print(f"Registrierte Domain: {registered_domain}")

Ziel-URL: https://zero.webappsecurity.com
Basis-URL: https://zero.webappsecurity.com
Registrierte Domain: webappsecurity.com


/tmp/ipykernel_91118/2463526908.py:8: DeprecationWarning: The 'registered_domain' property is deprecated and will be removed in the next major version. Use 'top_domain_under_public_suffix' instead, which has the same behavior but a more accurate name.
  registered_domain = tldextract.extract(url).registered_domain


In [69]:
import ssl
import warnings
import urllib3
from requests.adapters import HTTPAdapter

warnings.filterwarnings("ignore", category=urllib3.exceptions.InsecureRequestWarning)


class LegacySSLAdapter(HTTPAdapter):
    """HTTPAdapter, der ältere TLS-Versionen (TLS 1.0/1.1) und schwache Cipher-Suiten erlaubt."""
    def init_poolmanager(self, *args, **kwargs):
        ctx = ssl.SSLContext(ssl.PROTOCOL_TLS_CLIENT)
        ctx.check_hostname = False
        ctx.verify_mode = ssl.CERT_NONE
        ctx.set_ciphers("DEFAULT:@SECLEVEL=0")
        ctx.minimum_version = ssl.TLSVersion.MINIMUM_SUPPORTED
        kwargs["ssl_context"] = ctx
        super().init_poolmanager(*args, **kwargs)


def create_chrome_driver(headless: bool = True) -> webdriver.Chrome:
    options = Options()
    if headless:
        options.add_argument("--headless=new")
    options.add_argument("--window-size=1920,1080")
    options.add_argument("--disable-popup-blocking")
    options.add_argument("--ignore-certificate-errors")
    options.add_argument("--ignore-ssl-errors")
    options.add_argument("--allow-running-insecure-content")
    options.add_argument("--allow-insecure-localhost")
    options.add_argument("--incognito")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--no-sandbox")
    options.set_capability("goog:loggingPrefs", {"performance": "ALL", "browser": "ALL"})
    return webdriver.Chrome(options=options)


def collect_browser_data_fallback(target_url: str) -> dict:
    """Fallback via requests: versucht Legacy-SSL-Adapter, dann HTTP-Downgrade.
    Performance-Logs sind in diesem Modus nicht verfügbar."""
    session = requests.Session()
    session.mount("https://", LegacySSLAdapter())
    session.headers.update({
        "User-Agent": (
            "Mozilla/5.0 (X11; Linux x86_64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/120.0.0.0 Safari/537.36"
        )
    })

    # Kandidaten: zuerst Original-URL, dann HTTP-Downgrade falls HTTPS
    candidates = [target_url]
    if target_url.lower().startswith("https://"):
        candidates.append("http://" + target_url[8:])

    last_exc = None
    for try_url in candidates:
        try:
            response = session.get(try_url, timeout=REQUEST_TIMEOUT, verify=False, allow_redirects=True)
            soup = BeautifulSoup(response.text, "html.parser")
            title = soup.title.string.strip() if soup.title and soup.title.string else ""
            if try_url != target_url:
                print(f"  HTTP-Downgrade verwendet: {try_url}")
            return {
                "title": title,
                "current_url": response.url,
                "page_source": response.text,
                "performance_logs": [],
                "browser_logs": [],
                "_fallback": True,
            }
        except Exception as exc:
            last_exc = exc

    raise last_exc


def collect_browser_data(target_url: str) -> dict:
    driver = create_chrome_driver(headless=True)
    try:
        driver.set_page_load_timeout(REQUEST_TIMEOUT)
        driver.get(target_url)
        time.sleep(PAGE_LOAD_WAIT_SECONDS)
        page_source = driver.page_source
        title = driver.title
        current_url = driver.current_url
        try:
            performance_logs = driver.get_log("performance")
        except Exception:
            performance_logs = []
        try:
            browser_logs = driver.get_log("browser")
        except Exception:
            browser_logs = []
    finally:
        driver.quit()

    return {
        "title": title,
        "current_url": current_url,
        "page_source": page_source,
        "performance_logs": performance_logs,
        "browser_logs": browser_logs,
    }


_SSL_ERRORS = (
    "ERR_SSL_VERSION_OR_CIPHER_MISMATCH",
    "ERR_SSL_PROTOCOL_ERROR",
    "ERR_SSL_",
    "ERR_CERT_",
    "net::ERR_",
)

try:
    browser_data = collect_browser_data(url)
    print(f"Browser-Daten gesammelt. Seitentitel: {browser_data['title']}")
    print(f"Performance-Log-Einträge: {len(browser_data['performance_logs'])}")
except WebDriverException as exc:
    exc_msg = str(exc)
    if any(marker in exc_msg for marker in _SSL_ERRORS):
        print(
            f"Hinweis: Chrome meldet ein SSL/TLS-Problem mit {url}:\n"
            f"  {exc_msg.splitlines()[0]}\n"
            "Weiche auf requests-Fallback aus (Legacy-SSL + HTTP-Downgrade) ..."
        )
        try:
            browser_data = collect_browser_data_fallback(url)
            print(f"Fallback erfolgreich. Seitentitel: '{browser_data['title']}'")
            print("Hinweis: Performance-Logs sind im Fallback-Modus nicht verfügbar.")
        except Exception as fallback_exc:
            raise RuntimeError(
                f"Selenium und requests-Fallback sind beide fehlgeschlagen.\n"
                f"Selenium-Fehler : {exc_msg.splitlines()[0]}\n"
                f"Fallback-Fehler : {fallback_exc}"
            ) from fallback_exc
    else:
        raise RuntimeError(
            f"Selenium-Fehler: {exc_msg[:500]}\n\n"
            "Prüfen Sie, ob Chrome/Chromium installiert ist (z.B. 'which chromium-browser')."
        ) from exc

Hinweis: Chrome meldet ein SSL/TLS-Problem mit https://zero.webappsecurity.com:
  Message: unknown error: net::ERR_SSL_VERSION_OR_CIPHER_MISMATCH
Weiche auf requests-Fallback aus (Legacy-SSL + HTTP-Downgrade) ...
  HTTP-Downgrade verwendet: http://zero.webappsecurity.com
Fallback erfolgreich. Seitentitel: 'Zero - Personal Banking - Loans - Credit Cards'
Hinweis: Performance-Logs sind im Fallback-Modus nicht verfügbar.


In [70]:
def normalize_headers(headers: dict) -> dict:
    return {str(k).lower(): v for k, v in dict(headers).items()}


def fetch_http_metadata(target_url: str) -> dict:
    session = requests.Session()
    session.headers.update({"User-Agent": "LocalSecurityNotebook/1.0"})
    result = {"url": target_url, "ok": False, "error": None, "final_url": None, "status_code": None, "headers": {}}
    try:
        response = session.get(target_url, timeout=REQUEST_TIMEOUT, allow_redirects=True)
        result.update({
            "ok": True,
            "final_url": response.url,
            "status_code": response.status_code,
            "headers": normalize_headers(response.headers),
        })
    except Exception as exc:
        result["error"] = str(exc)
    return result


def fetch_well_known(base_url: str) -> dict:
    paths = ["/robots.txt", "/.well-known/security.txt"]
    results = {}
    for path in paths:
        full_url = urljoin(base_url, path)
        try:
            r = requests.get(full_url, timeout=REQUEST_TIMEOUT, allow_redirects=True)
            text_preview = r.text[:1500] if r.text else ""
            results[path] = {
                "url": full_url,
                "status_code": r.status_code,
                "content_type": r.headers.get("content-type", ""),
                "body_preview": text_preview,
            }
        except Exception as exc:
            results[path] = {"url": full_url, "error": str(exc)}
    return results

http_metadata = fetch_http_metadata(url)
well_known = fetch_well_known(base_url)

print(json.dumps({
    "status_code": http_metadata.get("status_code"),
    "final_url": http_metadata.get("final_url"),
    "robots": well_known.get("/robots.txt", {}).get("status_code"),
    "security_txt": well_known.get("/.well-known/security.txt", {}).get("status_code"),
}, indent=2, ensure_ascii=False))

{
  "status_code": null,
  "final_url": null,
  "robots": null,
  "security_txt": null
}


In [71]:
def parse_performance_logs(logs: list) -> dict:
    requests_seen = []
    responses_seen = []
    resources = Counter()
    domains = Counter()
    insecure_requests = []
    mixed_content = []

    for entry in logs:
        try:
            message = json.loads(entry.get("message", "{}"))["message"]
            method = message.get("method", "")
            params = message.get("params", {})
        except Exception:
            continue

        if method == "Network.requestWillBeSent":
            req = params.get("request", {})
            req_url = req.get("url", "")
            document_url = params.get("documentURL", "")
            if req_url:
                requests_seen.append({
                    "url": req_url,
                    "method": req.get("method"),
                    "headers": req.get("headers", {}),
                    "type": params.get("type"),
                    "initiator": params.get("initiator", {}).get("url"),
                })
                domain = urlparse(req_url).netloc
                if domain:
                    domains[domain] += 1
                if req_url.startswith("http://"):
                    insecure_requests.append(req_url)
                if document_url.startswith("https://") and req_url.startswith("http://"):
                    mixed_content.append(req_url)

        if method == "Network.responseReceived":
            response = params.get("response", {})
            resource_type = params.get("type", "Unknown")
            resources[resource_type] += 1
            responses_seen.append({
                "url": response.get("url", ""),
                "status": response.get("status"),
                "mimeType": response.get("mimeType"),
                "headers": response.get("headers", {}),
                "securityDetails": response.get("securityDetails", {}),
            })

    return {
        "request_count": len(requests_seen),
        "response_count": len(responses_seen),
        "resource_counts": dict(resources),
        "top_domains": domains.most_common(15),
        "insecure_requests": sorted(set(insecure_requests))[:50],
        "mixed_content": sorted(set(mixed_content))[:50],
        "responses": responses_seen[:200],
        "requests": requests_seen[:200],
    }

network_data = parse_performance_logs(browser_data["performance_logs"])
print(json.dumps({
    "requests": network_data["request_count"],
    "responses": network_data["response_count"],
    "resources": network_data["resource_counts"],
    "top_domains": network_data["top_domains"][:5],
    "insecure_requests": len(network_data["insecure_requests"]),
    "mixed_content": len(network_data["mixed_content"]),
}, indent=2, ensure_ascii=False))

{
  "requests": 0,
  "responses": 0,
  "resources": {},
  "top_domains": [],
  "insecure_requests": 0,
  "mixed_content": 0
}


In [72]:
SECURITY_HEADERS = {
    "strict-transport-security": "HTTP Strict Transport Security schützt vor Downgrade- und bestimmten Man-in-the-Middle-Risiken.",
    "content-security-policy": "Content Security Policy reduziert Risiken durch XSS und nicht vertrauenswürdige Skripte.",
    "x-content-type-options": "X-Content-Type-Options: nosniff reduziert MIME-Sniffing-Risiken.",
    "x-frame-options": "X-Frame-Options schützt gegen Clickjacking, wird teilweise durch CSP frame-ancestors ersetzt.",
    "referrer-policy": "Referrer-Policy steuert, welche Referrer-Informationen an Zielseiten weitergegeben werden.",
    "permissions-policy": "Permissions-Policy beschränkt Browser-Funktionen wie Kamera, Mikrofon oder Geolocation.",
}


def analyze_headers(headers: dict, scheme: str) -> list:
    findings = []
    for header, explanation in SECURITY_HEADERS.items():
        if header not in headers:
            severity = "Mittel"
            if header in {"strict-transport-security", "content-security-policy"}:
                severity = "Hoch" if scheme == "https" else "Mittel"
            findings.append({
                "category": "Security Header",
                "severity": severity,
                "finding": f"Fehlender Header: {header}",
                "evidence": "Header wurde in der HTTP-Antwort nicht gefunden.",
                "recommendation": explanation,
            })

    csp = headers.get("content-security-policy", "")
    if csp and ("unsafe-inline" in csp or "unsafe-eval" in csp):
        findings.append({
            "category": "Content Security Policy",
            "severity": "Mittel",
            "finding": "CSP enthält potenziell unsichere Direktiven.",
            "evidence": csp[:700],
            "recommendation": "Prüfen Sie, ob unsafe-inline oder unsafe-eval entfernt und durch Nonces, Hashes oder strengere Quellen ersetzt werden können.",
        })

    xcto = headers.get("x-content-type-options", "")
    if xcto and xcto.lower() != "nosniff":
        findings.append({
            "category": "Security Header",
            "severity": "Niedrig",
            "finding": "X-Content-Type-Options ist nicht auf nosniff gesetzt.",
            "evidence": xcto,
            "recommendation": "Setzen Sie X-Content-Type-Options auf nosniff.",
        })
    return findings


def analyze_html(source: str, target_url: str) -> dict:
    soup = BeautifulSoup(source, "html.parser")
    scripts = soup.find_all("script")
    forms = soup.find_all("form")
    inputs = soup.find_all("input")
    links = soup.find_all("a")
    target_domain = tldextract.extract(target_url).registered_domain
    third_party_scripts = []
    inline_script_count = 0

    for script in scripts:
        src = script.get("src")
        if src:
            full = urljoin(target_url, src)
            script_domain = tldextract.extract(full).registered_domain
            if script_domain and script_domain != target_domain:
                third_party_scripts.append(full)
        else:
            inline_script_count += 1

    form_summaries = []
    for form in forms:
        method = (form.get("method") or "GET").upper()
        action = form.get("action") or ""
        has_password = bool(form.find("input", attrs={"type": "password"}))
        form_summaries.append({"method": method, "action": urljoin(target_url, action), "has_password": has_password})

    return {
        "title": soup.title.string.strip() if soup.title and soup.title.string else "",
        "counts": {
            "links": len(links), "inputs": len(inputs), "forms": len(forms), "scripts": len(scripts),
            "inline_scripts": inline_script_count, "third_party_scripts": len(third_party_scripts),
        },
        "third_party_scripts": sorted(set(third_party_scripts))[:50],
        "forms": form_summaries,
    }


def build_findings() -> list:
    findings = []
    parsed_final = urlparse(http_metadata.get("final_url") or url)
    headers = http_metadata.get("headers", {}) or {}

    if parsed_final.scheme != "https":
        findings.append({"category": "Transport Security", "severity": "Hoch", "finding": "Die finale URL verwendet kein HTTPS.", "evidence": http_metadata.get("final_url") or url, "recommendation": "Erzwingen Sie HTTPS und leiten Sie HTTP dauerhaft auf HTTPS um."})

    findings.extend(analyze_headers(headers, parsed_final.scheme))

    for req in network_data.get("insecure_requests", []):
        findings.append({"category": "Transport Security", "severity": "Hoch", "finding": "Unsichere HTTP-Anfrage erkannt.", "evidence": req, "recommendation": "Laden Sie alle Ressourcen über HTTPS oder entfernen Sie die Ressource."})

    for req in network_data.get("mixed_content", []):
        findings.append({"category": "Mixed Content", "severity": "Hoch", "finding": "Mixed Content erkannt.", "evidence": req, "recommendation": "Vermeiden Sie HTTP-Ressourcen auf HTTPS-Seiten."})

    html_analysis = analyze_html(browser_data["page_source"], url)
    if html_analysis["counts"]["inline_scripts"] > 0:
        findings.append({"category": "JavaScript Security", "severity": "Mittel", "finding": "Inline-JavaScript wurde gefunden.", "evidence": f"Anzahl Inline-Skripte: {html_analysis['counts']['inline_scripts']}", "recommendation": "Reduzieren Sie Inline-Skripte und verwenden Sie eine strikte CSP mit Nonces oder Hashes."})

    if html_analysis["counts"]["third_party_scripts"] > 0:
        findings.append({"category": "Third-Party Scripts", "severity": "Mittel", "finding": "Externe Skripte von Dritt-Domains wurden gefunden.", "evidence": html_analysis["third_party_scripts"][:10], "recommendation": "Prüfen Sie Drittanbieter-Skripte regelmäßig, verwenden Sie Subresource Integrity wo möglich und beschränken Sie Quellen per CSP."})

    for form in html_analysis["forms"]:
        if form["has_password"] and form["method"] != "POST":
            findings.append({"category": "Form Security", "severity": "Hoch", "finding": "Passwortformular verwendet nicht POST.", "evidence": form, "recommendation": "Passwortformulare sollten POST verwenden und ausschließlich über HTTPS übertragen werden."})

    security_txt = well_known.get("/.well-known/security.txt", {})
    if security_txt.get("status_code") not in {200, 204}:
        findings.append({"category": "Security Contact", "severity": "Niedrig", "finding": "security.txt wurde nicht gefunden.", "evidence": security_txt, "recommendation": "Veröffentlichen Sie /.well-known/security.txt nach RFC 9116, damit Sicherheitsforscher Kontaktinformationen finden."})

    robots_txt = well_known.get("/robots.txt", {})
    if robots_txt.get("status_code") == 200:
        body = robots_txt.get("body_preview", "")
        if re.search(r"(?i)(admin|backup|private|secret|test|dev|staging)", body):
            findings.append({"category": "Information Disclosure", "severity": "Mittel", "finding": "robots.txt enthält potenziell sensible Pfadhinweise.", "evidence": body[:700], "recommendation": "robots.txt sollte keine sensiblen oder nicht geschützten Bereiche offenlegen. Zugriffsschutz muss serverseitig erfolgen."})

    return findings, html_analysis

findings, html_analysis = build_findings()
print(f"Ermittelte Findings: {len(findings)}")
print(json.dumps(findings[:5], indent=2, ensure_ascii=False))

Ermittelte Findings: 8
[
  {
    "category": "Security Header",
    "severity": "Hoch",
    "finding": "Fehlender Header: strict-transport-security",
    "evidence": "Header wurde in der HTTP-Antwort nicht gefunden.",
    "recommendation": "HTTP Strict Transport Security schützt vor Downgrade- und bestimmten Man-in-the-Middle-Risiken."
  },
  {
    "category": "Security Header",
    "severity": "Hoch",
    "finding": "Fehlender Header: content-security-policy",
    "evidence": "Header wurde in der HTTP-Antwort nicht gefunden.",
    "recommendation": "Content Security Policy reduziert Risiken durch XSS und nicht vertrauenswürdige Skripte."
  },
  {
    "category": "Security Header",
    "severity": "Mittel",
    "finding": "Fehlender Header: x-content-type-options",
    "evidence": "Header wurde in der HTTP-Antwort nicht gefunden.",
    "recommendation": "X-Content-Type-Options: nosniff reduziert MIME-Sniffing-Risiken."
  },
  {
    "category": "Security Header",
    "severity": "Mittel

/tmp/ipykernel_91118/1670394448.py:54: DeprecationWarning: The 'registered_domain' property is deprecated and will be removed in the next major version. Use 'top_domain_under_public_suffix' instead, which has the same behavior but a more accurate name.
  target_domain = tldextract.extract(target_url).registered_domain
/tmp/ipykernel_91118/1670394448.py:62: DeprecationWarning: The 'registered_domain' property is deprecated and will be removed in the next major version. Use 'top_domain_under_public_suffix' instead, which has the same behavior but a more accurate name.
  script_domain = tldextract.extract(full).registered_domain


In [73]:
def call_ollama(prompt: str, host: str = OLLAMA_HOST, model: str = OLLAMA_MODEL) -> str:
    payload = {
        "model": model,
        "prompt": prompt,
        "stream": False,
        "options": {"temperature": 0.2, "num_ctx": 8192},
    }
    response = requests.post(f"{host}/api/generate", json=payload, timeout=180)
    response.raise_for_status()
    return response.json().get("response", "")


def build_report_prompt() -> str:
    evidence = {
        "target_url": url,
        "final_url": http_metadata.get("final_url"),
        "status_code": http_metadata.get("status_code"),
        "page_title": browser_data.get("title"),
        "html_analysis": html_analysis,
        "network_summary": {
            "request_count": network_data.get("request_count"),
            "response_count": network_data.get("response_count"),
            "resource_counts": network_data.get("resource_counts"),
            "top_domains": network_data.get("top_domains"),
            "insecure_requests": network_data.get("insecure_requests"),
            "mixed_content": network_data.get("mixed_content"),
        },
        "http_headers": http_metadata.get("headers"),
        "well_known": well_known,
        "findings": findings,
    }
    evidence_text = json.dumps(evidence, indent=2, ensure_ascii=False)[:MAX_PROMPT_CHARS]
    return f"""
Sie sind ein erfahrener Cybersecurity Analyst. Erstellen Sie auf Basis der folgenden technischen Beobachtungen einen professionellen, klar strukturierten Website-Security-Report in Deutsch.

Wichtige Regeln:
- Nur Findings aufführen, die durch die bereitgestellten Beobachtungen gestützt sind.
- Keine Exploit-Anleitungen und keine aktiven Angriffsschritte beschreiben.
- Risiken realistisch bewerten und keine Schwachstellen erfinden.
- Empfehlungen sollen praktisch, priorisiert und umsetzbar sein.
- Der Report soll Management-tauglich und gleichzeitig technisch nützlich sein.

Gewünschte Struktur:
# Website Security Report: <Domain>
## 1. Kurzfazit
## 2. Scope und Methodik
## 3. Priorisierte Findings
Für jedes Finding: Schweregrad, Beobachtung, Risiko, Empfehlung.
## 4. Positive Beobachtungen
## 5. Empfohlene nächste Schritte
## 6. Hinweis zur Aussagekraft

Technische Beobachtungen:
```json
{evidence_text}
```
""".strip()

prompt = build_report_prompt()
print(f"Prompt-Länge: {len(prompt)} Zeichen")

Prompt-Länge: 5240 Zeichen


In [74]:
report = call_ollama(prompt)
display(Markdown(report))

 # Website Security Report: zero.webappsecurity.com

## 1. Kurzfazit
Die Analyse ergab eine Reihe von Sicherheitsfindings, die hauptsächlich auf fehlende oder fehlerhafte HTTP-Headers zurückgehen. Es wurden keine aktiven Angriffsschritte durchgeführt und keine Schwachstellen erfunden. Die wichtigsten Findings betreffen den Fehlen der Headers `strict-transport-security`, `content-security-policy`, `x-content-type-options`, `x-frame-options`, `referrer-policy`, `permissions-policy` und das Vorhandensein von Inline-JavaScript.

## 2. Scope und Methodik
Die Analyse basiert auf einer automatisierten Scanung der Website zero.webappsecurity.com, die auf HTTP-Headers, HTML-Analyse, Netzwerkverbindungen und Well-Known-Dateien (`.well-known/security.txt`) fokussiert ist. Die Analyse wurde durchgeführt, indem eine Reihe von Tools verwendet wurde, um die Website zu scannen und die Ergebnisse zu sammeln.

## 3. Priorisierte Findings
Für jedes Finding wird der Schweregrad, die Beobachtung, das Risiko und die Empfehlung angegeben.

### Schweregrad: Hoch
- Fehlernder Header: `strict-transport-security`
- Fehlernder Header: `content-security-policy`

### Schweregrad: Mittel
- Fehlernder Header: `x-content-type-options`
- Fehlernder Header: `x-frame-options`
- Fehlernder Header: `referrer-policy`
- Fehlernder Header: `permissions-policy`
- Inline-JavaScript wurde gefunden

### Schweregrad: Niedrig
- `security.txt` wurde nicht gefunden

## 4. Positive Beobachtungen
Es wurden keine positiven Beobachtungen gemacht.

## 5. Empfohlene nächste Schritte
Um die Sicherheit der Website zu verbessern, sollten die folgenden Schritte durchgeführt werden:
- Implementieren Sie HTTP Strict Transport Security (HSTS) und Content Security Policy (CSP).
- Reduzieren Sie Inline-JavaScript und verwenden Sie eine strikte CSP mit Nonces oder Hashes.
- Veröffentlichen Sie /.well-known/security.txt nach RFC 9116, damit Sicherheitsforscher Kontaktinformationen finden.

## 6. Hinweis zur Aussagekraft
Die Analyse basiert auf der automatisierten Scanung der Website und kann möglicherweise nicht alle Schwachstellen erkennen oder detektieren. Es wird empfohlen, die Ergebnisse durch eine Manual-Review zu bestätigen und weitere Sicherheitsmaßnahmen durchzuführen, um die Gesamtsicherheit der Website sicherzustellen.

In [75]:
# Optional: Report als Markdown-Datei speichern
safe_domain = re.sub(r"[^a-zA-Z0-9_.-]", "_", registered_domain or "website")
output_file = f"security_report_{safe_domain}.md"

with open(output_file, "w", encoding="utf-8") as f:
    f.write(report)

print(f"Report gespeichert: {output_file}")

Report gespeichert: security_report_webappsecurity.com.md
